# RetainIQ — Phase 8.1: Benchmark Dataset & Benchmark Inputs

## Objective

I establish the benchmark reference layer for RetainIQ. I use the executed Phase 7 outputs as the business source of truth and create transparent internal benchmarks before introducing any optional external benchmark.

The internal benchmarks are:

- **Portfolio benchmark:** weighted churn rate across the complete customer base represented by the Phase 7 segment table.
- **Peer-market benchmark:** median churn rate across qualifying city markets.
- **Value/exposure references:** portfolio average CLTV and historical revenue-at-risk per customer.

External benchmarking is optional and is only used when I provide a directly comparable benchmark value and documented source.

### Notebook presentation rule

For every relevant analytical code cell, I place a **Result & conclusion** markdown cell immediately after it. Setup-only cells are not padded with artificial conclusions.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

ROOT = Path("..")
OUTPUT_DIR = ROOT / "outputs"
CONFIG_DIR = ROOT / "config"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def load_csv(filename):
    path = OUTPUT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Required Phase 7 output not found: {path}. "
            "Run Phase 7.1–7.4 first."
        )
    return pd.read_csv(path)


def print_result(message):
    print(f"Result: {message}")

In [3]:
segment_df = load_csv(r"C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\Phase 7 — Retention Strategy & Business Intelligence Layer\outputs\phase_07_segment_business_base.csv")
state_df = load_csv(r"C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\Phase 7 — Retention Strategy & Business Intelligence Layer\outputs\phase_07_state_business_base.csv")
city_df = load_csv(r"C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\Phase 7 — Retention Strategy & Business Intelligence Layer\outputs\phase_07_city_business_base.csv")
segment_priority = load_csv(r"C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\Phase 7 — Retention Strategy & Business Intelligence Layer\outputs\segment_priority.csv")
city_priority = load_csv(r"C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\Phase 7 — Retention Strategy & Business Intelligence Layer\outputs\city_priority.csv")

required_segment_cols = {"segment_id", "segment_name", "customers", "churned_customers", "churn_rate_pct", "avg_cltv", "revenue_at_risk"}
required_state_cols = {"state", "customers", "churned_customers", "churn_rate_pct", "avg_cltv", "revenue_at_risk"}
required_city_cols = {"state", "city", "customers", "churned_customers", "churn_rate_pct", "avg_cltv", "revenue_at_risk"}

for frame, required, name in [
    (segment_df, required_segment_cols, "segment_df"),
    (state_df, required_state_cols, "state_df"),
    (city_df, required_city_cols, "city_df"),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

print_result(
    f"Loaded Phase 7 business outputs: {len(segment_df)} segments, {len(state_df)} states, and {len(city_df)} state/city markets."
)

Result: Loaded Phase 7 business outputs: 3 segments, 1 states, and 1106 state/city markets.


### Result & conclusion

The Phase 7 business-analysis outputs are available at their intended grains. I can now construct benchmarks without rebuilding earlier transformations.

In [4]:
numeric_map = {
    "customers": "int64",
    "churned_customers": "int64",
    "churn_rate_pct": "float64",
    "avg_cltv": "float64",
    "revenue_at_risk": "float64",
}

for frame in [segment_df, state_df, city_df, segment_priority, city_priority]:
    for col in numeric_map:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")

print_result("Numeric benchmark fields were converted to analysis-ready types.")

Result: Numeric benchmark fields were converted to analysis-ready types.


### Result & conclusion

The core benchmark measures are now numeric, which makes gap calculations and peer comparisons reproducible.

In [5]:
portfolio_customers = int(segment_df["customers"].sum())
portfolio_churned = int(segment_df["churned_customers"].sum())
portfolio_churn_rate = (portfolio_churned / portfolio_customers * 100) if portfolio_customers else np.nan
cltv_valid = segment_df.dropna(subset=["avg_cltv", "customers"]).copy()
portfolio_avg_cltv = (
    np.average(cltv_valid["avg_cltv"], weights=cltv_valid["customers"])
    if not cltv_valid.empty and cltv_valid["customers"].sum() > 0
    else np.nan
)
portfolio_revenue_at_risk = float(segment_df["revenue_at_risk"].sum())
portfolio_revenue_at_risk_per_customer = (portfolio_revenue_at_risk / portfolio_customers) if portfolio_customers else np.nan
city_params = pd.read_csv(CONFIG_DIR / "benchmark_parameters.csv")
city_min = int(city_params.loc[city_params["parameter"] == "minimum_city_market_size", "value"].iloc[0])
benchmark_tolerance_pp = float(city_params.loc[city_params["parameter"] == "benchmark_tolerance_pp", "value"].iloc[0])
qualifying_cities = city_df[city_df["customers"] >= city_min].copy()
peer_city_median_churn = qualifying_cities["churn_rate_pct"].median() if not qualifying_cities.empty else np.nan
peer_city_median_cltv = qualifying_cities["avg_cltv"].median() if not qualifying_cities.empty else np.nan
peer_city_median_revenue_at_risk = qualifying_cities["revenue_at_risk"].median() if not qualifying_cities.empty else np.nan
benchmark_reference = pd.DataFrame([
    ["portfolio", "Portfolio weighted churn", "churn_rate_pct", portfolio_churn_rate, "%", "Internal Phase 7 customer base", None, portfolio_customers, "Weighted by customer counts."],
    ["portfolio", "Portfolio weighted CLTV", "avg_cltv", portfolio_avg_cltv, "dataset currency", "Internal Phase 7 segment base", None, portfolio_customers, "Weighted by customer counts."],
    ["portfolio", "Portfolio revenue-at-risk per customer", "revenue_at_risk_per_customer", portfolio_revenue_at_risk_per_customer, "dataset currency/customer", "Internal Phase 7 segment base", None, portfolio_customers, "Historical revenue-at-risk proxy divided by total customers."],
    ["peer_city", "Median qualifying-city churn", "churn_rate_pct", peer_city_median_churn, "%", "Internal Phase 7 city base", None, len(qualifying_cities), f"Median across cities with at least {city_min} customers."],
    ["peer_city", "Median qualifying-city CLTV", "avg_cltv", peer_city_median_cltv, "dataset currency", "Internal Phase 7 city base", None, len(qualifying_cities), f"Median across cities with at least {city_min} customers."],
    ["peer_city", "Median qualifying-city revenue at risk", "revenue_at_risk", peer_city_median_revenue_at_risk, "dataset currency", "Internal Phase 7 city base", None, len(qualifying_cities), f"Median across cities with at least {city_min} customers."],
], columns=["benchmark_scope","benchmark_name","metric","benchmark_value","unit","source","source_date","population","notes"])
print_result(
    f"Portfolio churn benchmark is {portfolio_churn_rate:.2f}% and the qualifying-city peer benchmark is {peer_city_median_churn:.2f}% using {len(qualifying_cities)} cities."
)
display(benchmark_reference)


Result: Portfolio churn benchmark is 26.54% and the qualifying-city peer benchmark is 26.62% using 27 cities.


,benchmark_scope,benchmark_name,metric,benchmark_value,unit,source,population,notes
0,portfolio,Portfolio weighted churn,churn_rate_pct,26.536987,%,Internal Phase 7 customer base,7043,Weighted by customer counts.
1,portfolio,Portfolio weighted CLTV,avg_cltv,4400.295752,dataset currency,Internal Phase 7 segment base,7043,Weighted by customer counts.
2,portfolio,Portfolio revenue-at-risk per customer,revenue_at_risk_per_customer,523.137842,dataset currency/customer,Internal Phase 7 segment base,7043,Historical revenue-at-risk proxy divided by to...
3,peer_city,Median qualifying-city churn,churn_rate_pct,26.620000,%,Internal Phase 7 city base,27,Median across cities with at least 25 customers.
4,peer_city,Median qualifying-city CLTV,avg_cltv,4388.860000,dataset currency,Internal Phase 7 city base,27,Median across cities with at least 25 customers.
5,peer_city,Median qualifying-city revenue at risk,revenue_at_risk,23726.130000,dataset currency,Internal Phase 7 city base,27,Median across cities with at least 25 customers.


### Result & conclusion

I now have explicit internal reference points for churn, customer value, and revenue exposure. These benchmarks are calculated from the project data itself and therefore do not require an external assumption.

In [6]:
external_path = CONFIG_DIR / "external_benchmark_inputs.csv"
external_df = pd.read_csv(external_path) if external_path.exists() else pd.DataFrame()

if not external_df.empty:
    external_df["benchmark_value"] = pd.to_numeric(external_df["benchmark_value"], errors="coerce")
    external_ready = external_df.dropna(subset=["benchmark_value"]).copy()
else:
    external_ready = pd.DataFrame()

print_result(
    f"External benchmark template contains {len(external_ready)} populated benchmark values; internal benchmarking will remain active regardless."
)
display(external_df)

Result: External benchmark template contains 0 populated benchmark values; internal benchmarking will remain active regardless.


,benchmark_scope,benchmark_name,metric,benchmark_value,unit,source,source_date,population,notes
0,external,Optional external benchmark,churn_rate_pct,NaN,NaN,Enter only a directly comparable benchmark.,NaN,NaN,Do not enter a value until the benchmark uses ...
1,external,Optional external benchmark,avg_monthly_charge,NaN,NaN,Enter only if the external source is directly ...,NaN,NaN,"Keep currency, customer population, and period..."


### Result & conclusion

External benchmarking remains optional. I will only compare against populated values, and I will retain the source, date, population, and notes so the comparison is traceable.

In [7]:
benchmark_reference.to_csv(OUTPUT_DIR / "benchmark_reference.csv", index=False)
external_df.to_csv(OUTPUT_DIR / "phase_08_external_benchmark_inputs.csv", index=False)
segment_df.to_csv(OUTPUT_DIR / "phase_07_segment_business_base.csv", index=False)
state_df.to_csv(OUTPUT_DIR / "phase_07_state_business_base.csv", index=False)
city_df.to_csv(OUTPUT_DIR / "phase_07_city_business_base.csv", index=False)
segment_priority.to_csv(OUTPUT_DIR / "segment_priority.csv", index=False)
city_priority.to_csv(OUTPUT_DIR / "city_priority.csv", index=False)

print_result("Saved the internal benchmark reference table, external benchmark input snapshot, and Phase 7 input bases.")


Result: Saved the internal benchmark reference table and external benchmark input snapshot.


### Result & conclusion

The benchmark assumptions are now preserved as portable CSV outputs, making later gap analysis reproducible and auditable.